# PVOUT Raster Transformation — COG and Tiles

Converts the clipped PVOUT GeoTIFF from `transformation.ipynb` into:
- **COG** (Cloud Optimized GeoTIFF) in Web Mercator
- **Visual tiles** — colorized XYZ tiles for map display
- **Value tiles** — Terrain RGB encoded for client-side hover lookup

**Input:** `output/poa_PVOUT.tif` (from transformation.ipynb, WGS84)  
**Outputs:** `output/poa_PVOUT_cog.tif`, `output/tiles_pvout/`, `output/tiles_pvout_values/`, `output/pvout_metadata.json`

**Requires:** GDAL (`gdalwarp`, `gdal_translate`, `gdaldem`, `gdal2tiles.py`)

In [12]:
import subprocess
from pathlib import Path

# Paths (run from v2/)
OUTPUT_DIR = Path("output")
INPUT_DIR = Path("source")
IN_TIF = INPUT_DIR / "poa_PVOUT.tif"
OUT_3857 = OUTPUT_DIR / "poa_PVOUT_3857.tif"
OUT_COG = OUTPUT_DIR / "poa_PVOUT_cog.tif"
OUT_COLORIZED = OUTPUT_DIR / "poa_PVOUT_colorized.tif"
OUT_VALUE_ENCODED = OUTPUT_DIR / "poa_PVOUT_value_encoded.tif"
TILES_VISUAL = OUTPUT_DIR / "tiles_pvout"
TILES_VALUES = OUTPUT_DIR / "tiles_pvout_values"
DATA_DIR = Path("data")
COLORS_VISUAL = DATA_DIR / "pvout_visual_colors.txt"
COLORS_VALUE = DATA_DIR / "pvout_value_encoding_colors.txt"
METADATA_JSON = OUTPUT_DIR / "pvout_metadata.json"

# Terrain RGB encoding params (for frontend decode: value = (R*256² + G*256 + B) * scale + offset)
PVOUT_SCALE = 0.001
PVOUT_OFFSET = 0.0
PVOUT_UNIT = "kWh/kWp/day"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not IN_TIF.exists():
    raise FileNotFoundError(f"Run transformation.ipynb first. Expected: {IN_TIF}")

## 1. Reproject to Web Mercator (EPSG:3857)

Tiles require Web Mercator. Reproject the clipped WGS84 GeoTIFF.

In [13]:
subprocess.run([
    "gdalwarp", "-t_srs", "EPSG:3857", "-r", "near",
    str(IN_TIF), str(OUT_3857)
], check=True)
print("Reprojected to Web Mercator:", OUT_3857)

Processing poa_PVOUT.tif [1/1] : 0...10...20...30...40...50...60...70...80...90...100 - done.
Reprojected to Web Mercator: output/poa_PVOUT_3857.tif


## 2. Convert to COG

Use `RESAMPLING=AVERAGE` for numeric solar data.

In [14]:
subprocess.run([
    "gdal_translate", str(OUT_3857), str(OUT_COG),
    "-of", "COG",
    "-co", "COMPRESS=DEFLATE",
    "-co", "OVERVIEWS=AUTO",
    "-co", "RESAMPLING=AVERAGE"
], check=True)
print("Created COG:", OUT_COG)

Input file size is 55, 76
0...10...20...30...40...50...60...70...80...90...100 - done.
Created COG: output/poa_PVOUT_cog.tif


## 3. Colorize for visual tiles

Map PVOUT values to a gradient (blue → yellow → orange) for map display.

In [15]:
subprocess.run([
    "gdaldem", "color-relief", str(OUT_COG), str(COLORS_VISUAL), str(OUT_COLORIZED)
], check=True)
print("Colorized for display:", OUT_COLORIZED)

0...10...20...30...40...50...60...70...80...90...100 - done.
Colorized for display: output/poa_PVOUT_colorized.tif


## 4. Generate visual XYZ tiles

In [16]:
TILES_VISUAL.mkdir(parents=True, exist_ok=True)
subprocess.run([
    "gdal2tiles.py", "-r", "near", "-z", "8-15", "--xyz", "-w", "none",
    str(OUT_COLORIZED), str(TILES_VISUAL)
], check=True)
print("Visual tiles written to:", TILES_VISUAL)

Generating Base Tiles:


0...10...20...30...40...50...60...70...80...90...100 - done in 00:00:08.
0.

Generating Overview Tiles:


..10...20...30...40...50...60...70...80...90...100 - done.
Visual tiles written to: output/tiles_pvout


## 5. Value tiles (Terrain RGB encoding)

Encode PVOUT values in RGB so the frontend can decode: `value = (R*256² + G*256 + B) * scale + offset`

In [10]:
subprocess.run([
    "gdaldem", "color-relief", str(OUT_COG), str(COLORS_VALUE), str(OUT_VALUE_ENCODED)
], check=True)

TILES_VALUES.mkdir(parents=True, exist_ok=True)
subprocess.run([
    "gdal2tiles.py", "-r", "near", "-z", "8-15", "--xyz", "-w", "none",
    str(OUT_VALUE_ENCODED), str(TILES_VALUES)
], check=True)
print("Value tiles written to:", TILES_VALUES)

0...10...20...30...40...50...60...70...80...90...100 - done.


Generating Base Tiles:


0...10...20...30...40...50...60...70...80...90...100 - done in 00:00:07.
0..

Generating Overview Tiles:


.10...20...30...40...50...60...70...80...90...100 - done.
Value tiles written to: output/tiles_pvout_values


## 6. Write metadata for frontend

Store scale, offset, and unit so the frontend can decode value tiles.

In [11]:
import json

metadata = {
    "layer": "PVOUT",
    "description": "PV Output (kWh/kWp/day)",
    "encoding": {
        "type": "terrain_rgb",
        "scale": PVOUT_SCALE,
        "offset": PVOUT_OFFSET,
        "unit": PVOUT_UNIT,
        "decode_formula": "value = (R * 256 * 256 + G * 256 + B) * scale + offset"
    }
}
with open(METADATA_JSON, "w") as f:
    json.dump(metadata, f, indent=2)
print("Metadata written to:", METADATA_JSON)

Metadata written to: output/pvout_metadata.json
